<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇳🇱 Netherlands Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: gtfs.ovapi.nl · data.ndovloket.nl · Provider: DOVA (Dienst OV-aanbesteding) · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

**Netherlands GTFS Dataset**
- Provider: OVapi (open data initiative aggregating all Dutch public transport operators)
- Publisher: DOVA (Dienst OV-aanbesteding)
- Format: GTFS
- Dataset page: https://gtfs.ovapi.nl/nl/
- Download URL: https://gtfs.ovapi.nl/nl/gtfs-nl.zip
- Accessed: May 2026
- License: CC0 (Public Domain)

**Netherlands NeTEx Dataset**
- Provider: NDOV Loket (Nationaal Databank Openbaar Vervoer)
- Publisher: DOVA (Dienst OV-aanbesteding)
- Format: NeTEx (European EPIP Profile)
- Dataset page: https://data.ndovloket.nl/netex/epiap/
- Download URL: https://data.ndovloket.nl/netex/epiap/NeTEx_DOVA_epiap_2026-05-05.xml.gz
- Accessed: May 2026
- License: CC0 (Public Domain)

*No registration needed for downloading either dataset*

## Table of Contents

### Part 1: Netherlands GTFS Exploration
- Data Source
- Comment on the GTFS ZIP contents
- Comment on the loaded GTFS tables
- Comment on the stop hierarchy
- Filter to keep only parent stations
- Check the quality of the GTFS station-level subset
- Checking for invalid coordinates
- Check whether the GTFS station subset contains international stations
- Explore GTFS line-level fields
- Check GTFS route labels
- Check GTFS route labels before deduplication
- Explore GTFS calendar dates
- Build a GTFS calendar summary

### Part 2: NeTEx Exploration
- Peeking at the raw XML without parsing
- Extracting StopPlace elements from the NeTEx file
- Converting RD New Coordinates to WGS84
- Extract lines — tag analysis
- Important Finding: Structure of the Netherlands NeTEx Data

### Part 3: Station-Level Comparison between GTFS and NeTEx
- Checking for ID-based matching potential
- Checking ID overlap between GTFS and NeTEx
- Station matching: KDTree approach
- Result: Station-Level Comparison
- Conclusion: Station Comparison

# Part 1: Netherlands GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import gzip
import xml.etree.ElementTree as ET
from pyproj import Transformer
from scipy.spatial import cKDTree
import numpy as np

In [2]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "NL-20260505.gtfs.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/NL-20260505.gtfs.zip


In [3]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
5,stop_times.txt,976.37
4,shapes.txt,227.04
8,trips.txt,60.21
6,stops.txt,5.24
1,calendar_dates.txt,4.24
7,transfers.txt,2.98
3,routes.txt,0.17
0,agency.txt,0.00
2,feed_info.txt,0.00


## Comment

All the expected parts are present.

One important point is that there is no `calendar.txt` visible in the ZIP contents. This means the calendar logic will probably have to be built mainly from `calendar_dates.txt`, similar to the France case.

In [4]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [5]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (70678, 11)
routes: (2962, 9)
trips: (867501, 12)
calendar_dates: (284975, 3)


## Comment 


The dataset contains 70,678 stop rows, 2,962 route rows, 867,501 trip rows, and 284,975 calendar-date rows. This confirms that the Netherlands GTFS feed is a large national feed.

For now, I only load the core tables that are needed for the MVD comparison. I do not load `stop_times.txt` at this stage because it is the largest file in the ZIP and is not needed for the first station-level checks.

The absence of `calendar.txt` means that the calendar analysis will probably have to be based mainly on `calendar_dates.txt`, similar to the France case.


In [6]:
# Check stop hierarchy
print("location_type value counts:")
print(stops["location_type"].value_counts(dropna=False))

print("\nTotal stops:", len(stops))
print("Missing location_type:", stops["location_type"].isna().sum())

location_type value counts:
location_type
0    54851
1    15827
Name: count, dtype: int64

Total stops: 70678
Missing location_type: 0


## Comment

The Netherlands GTFS `stops` table has a clear station hierarchy.

There are 70,678 stop rows in total. Most rows have `location_type = 0`, which means they are normal stops or platforms. There are also 15,827 rows with `location_type = 1`, which represent parent stations.

There are no missing `location_type` values, so the hierarchy field is complete. 

This is similar to France and different from Luxembourg, where the GTFS stop hierarchy was flat.

## Filter to keep only parent stations

In [7]:
# Define station-level subset
gtfs_station_stops = stops[stops["location_type"] == 1].copy()

print("Station-level stops (location_type = 1):", len(gtfs_station_stops))

# Check missingness on core MVD fields
print("\nMissingness in core fields:")
print(gtfs_station_stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]].isna().sum())

Station-level stops (location_type = 1): 15827

Missingness in core fields:
stop_id      0
stop_name    0
stop_lat     0
stop_lon     0
dtype: int64


## Check the quality of the GTFS station-level subset

After filtering to parent stations, I check whether the station-level GTFS table is clean enough for later comparison.

For the MVD station level, the most important GTFS fields are:

- `stop_id`
- `stop_name`
- `stop_lat`
- `stop_lon`

I also check whether the parent station IDs are unique and whether the coordinates look complete.

In [8]:
# Core station-level fields
station_core_cols = ["stop_id", "stop_name", "stop_lat", "stop_lon"]

print("GTFS station-level subset shape:")
print(gtfs_station_stops.shape)

print("\nMissing values in core station fields:")
print(gtfs_station_stops[station_core_cols].isna().sum())

print("\nUnique stop_id values:")
print(gtfs_station_stops["stop_id"].nunique())

print("\nDuplicate stop_id values:")
print(gtfs_station_stops["stop_id"].duplicated().sum())

print("\nDuplicate stop_name values:")
print(gtfs_station_stops["stop_name"].duplicated().sum())

print("\nCoordinate ranges:")
print(gtfs_station_stops[["stop_lat", "stop_lon"]].describe())

display(gtfs_station_stops[station_core_cols].head())

GTFS station-level subset shape:
(15827, 11)

Missing values in core station fields:
stop_id      0
stop_name    0
stop_lat     0
stop_lon     0
dtype: int64

Unique stop_id values:
15827

Duplicate stop_id values:
0

Duplicate stop_name values:
850

Coordinate ranges:
           stop_lat      stop_lon
count  15827.000000  15827.000000
mean      52.063798      5.453071
std        1.546950      0.935718
min        0.000000     -0.125150
25%       51.805480      4.846095
50%       52.121818      5.430635
75%       52.465436      5.989580
max       55.672760     21.052040


,stop_id,stop_name,stop_lat,stop_lon
54851,stoparea:109705,Tongeren,50.784104,5.474005
54852,stoparea:109707,Liers,50.698169,5.568750
54853,stoparea:109708,Nijmegen Goffert,51.827271,5.822334
54854,stoparea:109709,Glons,50.750663,5.535193
54855,stoparea:109711,Bilzen,50.868661,5.509376


## Checking for Invalid Coordinates

The coordinate range check from the previous step revealed a minimum latitude of `0.0` and a minimum longitude of `-0.125`, which are suspicious values for a Netherlands feed. A latitude of exactly zero almost certainly means a missing value encoded as zero rather than `NaN`, and a negative longitude places a stop outside the Netherlands. I therefore run a targeted check to identify and inspect these stations.

In [9]:
# Check for suspicious coordinates
invalid_coords = gtfs_station_stops[
    (gtfs_station_stops["stop_lat"] == 0) | 
    (gtfs_station_stops["stop_lon"] < 0)
]
print(f"Stations with invalid coordinates: {len(invalid_coords)}")
display(invalid_coords[["stop_id", "stop_name", "stop_lat", "stop_lon"]])

Stations with invalid coordinates: 9


,stop_id,stop_name,stop_lat,stop_lon
55380,stoparea:279985,London St. Pancras Int.,51.53054,-0.12515
57530,stoparea:489942,Aschaffenburg,0.00000,0.00000
57531,stoparea:489943,Ringsheim/Europa-Park,0.00000,0.00000
57533,stoparea:489945,Wien Hütteldorf,0.00000,0.00000
57534,stoparea:489947,Salzburg Hbf,0.00000,0.00000
57536,stoparea:489949,Amstetten NÖ,0.00000,0.00000
57539,stoparea:489952,St.Pölten Hbf,0.00000,0.00000
63428,stoparea:589649,München Ost,0.00000,0.00000
63429,stoparea:589650,Straubing,0.00000,0.00000


## Comment

Only **9 out of 15,827 stations** have suspicious coordinates. **London St. Pancras International** has a negative longitude of `-0.125`, which is actually correct since London is west of the prime meridian. It appears in the feed because the Netherlands has direct Eurostar services to London.

The remaining **8 stations**, including Salzburg Hbf, Wien Hütteldorf, München Ost and others, have coordinates of exactly `0.0, 0.0`, which are clearly missing values encoded as zero. These are all German and Austrian stations served by international trains from the Netherlands.

Since these 9 stations represent only **0.05%** of the total station table and are all international cross-border stops, this issue will not affect the GTFS–NeTEx comparison in any meaningful way.

## Comment

The GTFS station-level subset is clean from a data-quality perspective.

There are 15,827 station-level rows, and all important MVD fields are complete: `stop_id`, `stop_name`, `stop_lat`, and `stop_lon`. The `stop_id` values are also unique, so each row represents one clear GTFS station-level object.

There are 850 duplicate station names. This is not automatically a problem, because different station objects can share the same public name, especially when the dataset contains multiple operators.

The coordinate range shows that the feed is not limited strictly to the Netherlands. Some stations are clearly outside the Netherlands. The first displayed rows already include Belgian stations such as Tongeren, Liers, Glons, and Bilzen. This suggests that the Dutch GTFS feed includes cross-border stations, which is expected for public transport data with international or regional cross-border services.

## Check whether the GTFS station subset contains international stations

The GTFS parent-station subset is technically clean, but the coordinate range shows that the feed includes stations outside the Netherlands.

Before using this table later in the MVD comparison, I briefly check how many stations fall inside a rough Netherlands coordinate range and how many are outside. This is only an exploratory check

In [10]:
# Rough coordinate range for the Netherlands
# This is only used for exploration, not as a strict country boundary.

nl_lat_min, nl_lat_max = 50.7, 53.8
nl_lon_min, nl_lon_max = 3.2, 7.4

gtfs_station_stops = gtfs_station_stops.copy()

gtfs_station_stops["roughly_in_netherlands"] = (
    gtfs_station_stops["stop_lat"].between(nl_lat_min, nl_lat_max)
    & gtfs_station_stops["stop_lon"].between(nl_lon_min, nl_lon_max)
)

print("Stations roughly inside the Netherlands:")
print(gtfs_station_stops["roughly_in_netherlands"].value_counts())

print("\nShare of stations roughly inside the Netherlands:")
print((gtfs_station_stops["roughly_in_netherlands"].value_counts(normalize=True) * 100).round(2))

print("\nExamples of stations outside the rough Netherlands range:")
display(
    gtfs_station_stops.loc[
        ~gtfs_station_stops["roughly_in_netherlands"],
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ].head(20)
)

Stations roughly inside the Netherlands:
roughly_in_netherlands
True     15658
False      169
Name: count, dtype: int64

Share of stations roughly inside the Netherlands:
roughly_in_netherlands
True     98.93
False     1.07
Name: proportion, dtype: float64

Examples of stations outside the rough Netherlands range:


,stop_id,stop_name,stop_lat,stop_lon
54852,stoparea:109707,Liers,50.698169,5.568750
54856,stoparea:109712,Herstal,50.660289,5.622210
54866,stoparea:124593,Milmort,50.693562,5.599448
54867,stoparea:130826,Rathenow,52.599770,12.354930
54869,stoparea:137134,Berlin Südkreuz,52.475468,13.365581
54904,stoparea:17791,Frankfurt (M) Hbf,50.107120,8.663830
54907,stoparea:17795,Altenberge,52.052245,7.481240
54913,stoparea:17802,Unna,51.539020,7.692410
54925,stoparea:17814,Hamm (Westf.),51.678039,7.807820
54926,stoparea:17815,Hamburg Hbf,53.552724,10.006970


## Comment

This geographic check shows that the Netherlands GTFS station-level subset is mainly located in the Netherlands area.

Out of 15,827 parent stations, 15,658 fall inside the rough Netherlands coordinate range. This is 98.93% of the station-level subset. Only 169 stations, or 1.07%, fall outside this rough range.

The examples outside the range include Belgian and German stations, such as Liers, Herstal, Berlin Südkreuz, Frankfurt, Hamburg Hbf, Dortmund Hbf, and Augsburg Hbf. This confirms that the Dutch GTFS feed contains some cross-border or international station references.

This is not a problem for the analysis. The important point is that the international part is small, while the dataset is clearly mainly focused on the Netherlands. 

## Explore GTFS line-level fields

After defining the GTFS station-level subset, I now inspect the GTFS line/route side.

For the line level, the most important GTFS table is `routes`.  
The main field for visible public comparison is usually `route_short_name`, but I first inspect the available columns before deciding which field is best for the Netherlands.

In [11]:
# Inspect GTFS route and trip columns

print("Routes columns:")
print(routes.columns.tolist())

print("\nTrips columns:")
print(trips.columns.tolist())

print("\nRoutes shape:", routes.shape)
print("Trips shape:", trips.shape)

display(routes.head())
display(trips.head())

Routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color', 'route_url']

Trips columns:
['route_id', 'service_id', 'trip_id', 'realtime_trip_id', 'trip_headsign', 'trip_short_name', 'trip_long_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'bikes_allowed']

Routes shape: (2962, 9)
Trips shape: (867501, 12)


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,route_url
0,134842,ALLGO,326,Lijn 326 (R-net) Almere Stad - Blaricum P+R,NaN,3,e00713,ffffff,NaN
1,134844,ALLGO,330,Lijn 330 (R-net) Almere Buiten - A'dam Bijlmer,NaN,3,e00713,ffffff,NaN
2,134845,ALLGO,N22,Lijn N22 (NightGo) A'dam Leidsepl.- Almere Buiten,NaN,3,f59c00,000000,NaN
3,134846,ALLGO,N23,Lijn N23 (NightGo) A'dam Centraal - Almere Cen...,NaN,3,e00713,ffffff,NaN
4,134847,ALLGO,M1,Lijn M1 (Havenmetro) Stat. Centrum - Haven Cen...,NaN,3,0067a7,ffffff,NaN


,route_id,service_id,trip_id,realtime_trip_id,trip_headsign,trip_short_name,trip_long_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,17562,692,260620735,IFF:IC:2835,Utrecht Centraal,2835,Intercity,0,NaN,1533206.0,0,1.0
1,17562,2227,260620765,IFF:IC:2825,Utrecht Centraal,2825,Intercity,0,NaN,1533206.0,0,1.0
2,17562,2278,260620779,IFF:IC:2830,Rotterdam Centraal,2830,Intercity,1,NaN,1533207.0,0,1.0
3,17562,691,260620801,IFF:IC:2828,Rotterdam Centraal,2828,Intercity,1,NaN,1533207.0,0,1.0
4,17562,692,260620844,IFF:IC:2831,Utrecht Centraal,2831,Intercity,0,NaN,1533206.0,0,1.0


## Comment

The GTFS line-level tables are available and contain the fields needed for the route comparison.

The `routes` table has 2,962 rows and includes the main line-level fields: `route_id`, `agency_id`, `route_short_name`, `route_long_name`, and `route_type`. The field `route_short_name` looks like the best public line label, because it contains visible route names such as `326`, `330`, `N22`, `N23`, and `M1`.

The `trips` table is much larger, with 867,501 rows. It links trips to routes through `route_id` and to service validity through `service_id`. This means it will be useful later for connecting the line-level and calendar-level parts of the analysis.

For the line comparison, the first practical GTFS line-side unit should be based on `route_short_name`, while keeping `route_id`, `agency_id`, `route_long_name`, and `route_type` for context.

## Check GTFS route labels

The `routes` table contains the GTFS line-level information.

Before using `route_short_name` as the public line label, I check whether this field is complete and how often the same visible label appears more than once. This is important because one public line label can sometimes correspond to several technical route rows.

In [12]:
# Check quality of GTFS route labels

print("Routes shape:")
print(routes.shape)

print("\nMissing route_short_name values:")
print(routes["route_short_name"].isna().sum())

print("\nUnique route_short_name values:")
print(routes["route_short_name"].nunique())

print("\nDuplicate route_short_name values:")
print(routes["route_short_name"].duplicated().sum())

print("\nRoute type counts:")
print(routes["route_type"].value_counts(dropna=False).sort_index())

# Check how many technical route rows exist per public label
gtfs_route_label_counts = (
    routes
    .groupby("route_short_name", dropna=False)
    .agg(
        n_route_ids=("route_id", "nunique"),
        n_agencies=("agency_id", "nunique"),
        example_agencies=("agency_id", lambda x: ", ".join(sorted(x.dropna().astype(str).unique())[:5])),
        example_long_names=("route_long_name", lambda x: " | ".join(x.dropna().astype(str).unique()[:3]))
    )
    .reset_index()
    .sort_values("n_route_ids", ascending=False)
)

display(gtfs_route_label_counts.head(20))

Routes shape:
(2962, 9)

Missing route_short_name values:
0

Unique route_short_name values:
1192

Duplicate route_short_name values:
1770

Route type counts:
route_type
0      47
1      14
2     162
3    2715
4      24
Name: count, dtype: int64


,route_short_name,n_route_ids,n_agencies,example_agencies,example_long_names
1152,Sprinter,58,2,"IFF:NS, IFF:R-NET_NS",Utrecht Centraal <-> Baarn SPR5500 | Utrecht C...
1156,Stopbus ipv trein,55,8,"IFF:ARRIVA, IFF:EUROBAHN, IFF:NMBS, IFF:NS, IF...",Stopbus ipv trein Aachen Hbf <-> Heerlen | Sto...
0,1,40,12,"ARR, BRAVO:ARR, BRENG, CXX, DELIJN",Malberg via MUMC - De Heeg | Hoensbroek via He...
102,2,39,13,"ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Oud Caberg via Heer - De Heeg | Brunssum/Hoens...
199,3,38,12,"ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Sittard via Lindenheuvel - Geleen | Wolder via...
1080,Intercity,37,1,IFF:NS,Rotterdam Centraal <-> Utrecht Centraal IC2800...
285,4,34,12,"ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Pottenberg/Caberg via Scharn - Valkenburg | Si...
365,5,31,12,"ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Daalhof via MUMC - Heugem | Heerlen via Landgr...
1142,Snelbus ipv trein,27,4,"IFF:ARRIVA, IFF:EUROBAHN, IFF:NS, IFF:VIAS",Snelbus ipv trein Boxmeer <-> Nijmegen | Snelb...
444,6,25,13,"ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Markt via De Geusselt - Amby | Kerkrade via Ey...


### Comment

The GTFS route table is usable for the line-level comparison.

There are 2,962 route rows, and `route_short_name` has no missing values. This means every route has a visible public label.

However, there are only 1,192 unique `route_short_name` values. This shows that many public route labels appear more than once in the raw `routes` table. Some labels, such as `Sprinter` and `Intercity`, are not unique line numbers. They describe a train service category. Therefore, the same label can appear many times in the GTFS `routes` table for different connections.

Because of this, the later GTFS–NeTEx line comparison should not be based only on the raw `route_id` rows. A fairer comparison will use deduplicated public labels from `route_short_name`, while keeping the route IDs only as additional information.

## Check GTFS route labels before deduplication


In [13]:
# Check whether the same GTFS public route label appears across different route types or agencies

route_type_names = {
    0: "Tram",
    1: "Metro",
    2: "Rail",
    3: "Bus",
    4: "Ferry",
    5: "Cable tram",
    6: "Aerial lift",
    7: "Funicular",
    11: "Trolleybus",
    12: "Monorail"
}

routes = routes.copy()

routes["route_type_name"] = (
    routes["route_type"]
    .map(route_type_names)
    .fillna(routes["route_type"].astype(str))
)

gtfs_route_label_check = (
    routes
    .groupby("route_short_name", dropna=False)
    .agg(
        n_route_rows=("route_id", "count"),
        n_route_ids=("route_id", "nunique"),
        n_agencies=("agency_id", "nunique"),
        n_route_types=("route_type_name", "nunique"),
        route_types=("route_type_name", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
        example_agencies=("agency_id", lambda x: ", ".join(sorted(x.dropna().astype(str).unique())[:5])),
        example_long_names=("route_long_name", lambda x: " | ".join(x.dropna().astype(str).unique()[:3]))
    )
    .reset_index()
    .sort_values(["n_route_types", "n_route_ids"], ascending=False)
)

print("Public labels used across more than one route type:")
print((gtfs_route_label_check["n_route_types"] > 1).sum())

print("\nPublic labels used across more than one agency:")
print((gtfs_route_label_check["n_agencies"] > 1).sum())

display(gtfs_route_label_check.head(20))

Public labels used across more than one route type:
36

Public labels used across more than one agency:
401


,route_short_name,n_route_rows,n_route_ids,n_agencies,n_route_types,route_types,example_agencies,example_long_names
103,20,11,11,8,3,"Bus, Ferry, Tram","BRAVO:CXX, CXX, DELIJN, HTM, QBUZZ",Veldhoven MMC - Airport - Best Station | Hulst...
126,22,9,9,8,3,"Bus, Ferry, Tram","ALLGO, DELIJN, GVB, HTM, QBUZZ",Lijn 22 (FlexiGo) Station Buiten - De Vaart | ...
0,1,40,40,12,2,"Bus, Tram","ARR, BRAVO:ARR, BRENG, CXX, DELIJN",Malberg via MUMC - De Heeg | Hoensbroek via He...
102,2,39,39,13,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Oud Caberg via Heer - De Heeg | Brunssum/Hoens...
199,3,38,38,12,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Sittard via Lindenheuvel - Geleen | Wolder via...
285,4,34,34,12,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Pottenberg/Caberg via Scharn - Valkenburg | Si...
365,5,31,31,12,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Daalhof via MUMC - Heugem | Heerlen via Landgr...
444,6,25,25,13,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Markt via De Geusselt - Amby | Kerkrade via Ey...
649,7,25,25,12,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",Forum MECC via Sibemaweg - Caberg/Pottenberg |...
1,10,22,22,11,2,"Bus, Tram","ARR, BRAVO:ARR, BRAVO:CXX, BRENG, CXX",P+R Noord via Centrum - Randwyck | Heerlen via...


## Comment

This output shows that `route_short_name` is useful, but not always enough on its own.

Some route labels are used more than once. For example, a simple label like `1`, `2`, or `103` can exist in different cities, for different operators, or even for different transport types.

The result shows that 36 labels appear in more than one transport type, for example bus and tram. It also shows that 401 labels appear for more than one agency. This matters because the same public label does not always mean the same route.


## Explore GTFS calendar dates

The GTFS file does not contain a `calendar.txt` table.  
Therefore, the calendar information has to be checked through `calendar_dates.txt`.

This table shows on which dates each `service_id` is active or changed. Before building any activity patterns, I first inspect the date range, the exception types, and how many service IDs are present.

In [14]:
# Inspect GTFS calendar_dates table

calendar_dates = calendar_dates.copy()

# Convert GTFS date format to normal datetime
calendar_dates["date_dt"] = pd.to_datetime(
    calendar_dates["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("calendar_dates shape:")
print(calendar_dates.shape)

print("\ncalendar_dates columns:")
print(calendar_dates.columns.tolist())

print("\nMissing values:")
print(calendar_dates[["service_id", "date", "exception_type", "date_dt"]].isna().sum())

print("\nDate range:")
print("Start:", calendar_dates["date_dt"].min())
print("End:", calendar_dates["date_dt"].max())

print("\nNumber of unique service_ids in calendar_dates:")
print(calendar_dates["service_id"].nunique())

print("\nException type counts:")
print(calendar_dates["exception_type"].value_counts(dropna=False).sort_index())

display(calendar_dates.head())

calendar_dates shape:
(284975, 4)

calendar_dates columns:
['service_id', 'date', 'exception_type', 'date_dt']

Missing values:
service_id        0
date              0
exception_type    0
date_dt           0
dtype: int64

Date range:
Start: 2026-05-04 00:00:00
End: 2026-12-12 00:00:00

Number of unique service_ids in calendar_dates:
4136

Exception type counts:
exception_type
1    284975
Name: count, dtype: int64


,service_id,date,exception_type,date_dt
0,1,20260504,1,2026-05-04
1,2,20260504,1,2026-05-04
2,2,20260601,1,2026-06-01
3,2,20260608,1,2026-06-08
4,3,20260504,1,2026-05-04


In [15]:
# Check service_id coverage between trips and calendar_dates

trips_service_ids = set(trips["service_id"].astype(str))
calendar_service_ids = set(calendar_dates["service_id"].astype(str))

print("Unique service_ids in trips:")
print(len(trips_service_ids))

print("\nUnique service_ids in calendar_dates:")
print(len(calendar_service_ids))

print("\nService_ids in trips also found in calendar_dates:")
print(len(trips_service_ids & calendar_service_ids))

print("\nService_ids in trips but not in calendar_dates:")
print(len(trips_service_ids - calendar_service_ids))

print("\nService_ids in calendar_dates but not in trips:")
print(len(calendar_service_ids - trips_service_ids))

Unique service_ids in trips:
4084

Unique service_ids in calendar_dates:
4136

Service_ids in trips also found in calendar_dates:
4084

Service_ids in trips but not in calendar_dates:
0

Service_ids in calendar_dates but not in trips:
52


## Comment

The GTFS calendar data looks clean and complete.

There are 284,975 rows in `calendar_dates`. The date range goes from 2026-05-04 to 2026-12-12. There are no missing values in `service_id`, `date`, or `exception_type`.

All rows have `exception_type = 1`. This means that `calendar_dates` directly lists active service dates. There are no removed dates in this table.

The service ID check also looks good. All 4,084 service IDs used in `trips` are also found in `calendar_dates`. This means the trips can be linked to calendar validity. There are 52 service IDs in `calendar_dates` that are not used in `trips`, but this is not a problem for the analysis.

Because there is no `calendar.txt`, the Netherlands calendar analysis should follow a similar approach to France: build service activity patterns directly from `calendar_dates`.

## Build a GTFS calendar summary

Since the GTFS file has no `calendar.txt`, the active dates are taken directly from `calendar_dates`.

For each `service_id`, I summarize how many active days it has and what its first and last active date are. This gives a first compact view of the GTFS validity patterns.

In [16]:
gtfs_calendar_summary = (
    calendar_dates[calendar_dates["exception_type"] == 1]
    .groupby("service_id")
    .agg(
        n_active_days=("date_dt", "nunique"),
        first_active_date=("date_dt", "min"),
        last_active_date=("date_dt", "max")
    )
    .reset_index()
    .sort_values("n_active_days", ascending=False)
)

print("GTFS calendar summary shape:")
print(gtfs_calendar_summary.shape)

print("\nActive day summary:")
print(gtfs_calendar_summary["n_active_days"].describe())

display(gtfs_calendar_summary.head(20))

GTFS calendar summary shape:
(4136, 4)

Active day summary:
count    4136.000000
mean       68.901112
std        65.467861
min         1.000000
25%        11.000000
50%        42.000000
75%       121.000000
max       223.000000
Name: n_active_days, dtype: float64


,service_id,n_active_days,first_active_date,last_active_date
1123,1124,223,2026-05-04,2026-12-12
794,795,222,2026-05-04,2026-12-12
351,352,222,2026-05-04,2026-12-12
1105,1106,222,2026-05-04,2026-12-12
1010,1011,222,2026-05-04,2026-12-12
1731,1732,222,2026-05-05,2026-12-12
507,508,222,2026-05-04,2026-12-12
1122,1123,222,2026-05-04,2026-12-11
1080,1081,222,2026-05-04,2026-12-12
1108,1109,222,2026-05-04,2026-12-12


## Comment

The GTFS calendar summary gives a first overview of the service patterns in the Netherlands feed.

There are 4,136 `service_id` values in `calendar_dates`. Each service has between 1 and 223 active days. This means that some services run only on very few dates, while others are active across almost the full feed period.

The longest-running services are active from the beginning of the feed window in May 2026 until December 2026. This fits with the earlier date range check.


# Part 2: NeTEx Exploration


The Netherlands NeTEx file is distributed as a `.xml.gz` file, a gzip-compressed XML document, rather than a ZIP archive like Luxembourg and France. This means I cannot use `zipfile.ZipFile` to open it. Instead, I use Python's built-in `gzip` library which decompresses the file on the fly as I read it.

Before extracting any data, I first check that the file exists and peek at the first few bytes to understand the structure of the XML without fully loading it into memory.

**About gzip:**
- Python Software Foundation (2026). *gzip — Support for gzip files*. Python 3 Documentation. https://docs.python.org/3/library/gzip.html

In [17]:
netex_path = data_dir / "NeTEx_DOVA_epiap_2026-05-05.xml.gz"

print("NeTEx exists:", netex_path.exists())
print(f"File size: {netex_path.stat().st_size / (1024*1024):.1f} MB")

# Peek at the first 5000 bytes without fully decompressing
with gzip.open(netex_path, "rb") as f:
    head = f.read(5000)

print(head.decode("utf-8", errors="replace"))

NeTEx exists: True
File size: 3.2 MB
<?xml version="1.0" encoding="UTF-8"?>
<PublicationDelivery xmlns="http://www.netex.org.uk/netex" xmlns:gml="http://www.opengis.net/gml/3.2" version="ntx:1.1">
  <PublicationTimestamp>2026-05-05T04:23:57.076</PublicationTimestamp>
  <ParticipantRef>DOVA</ParticipantRef>
  <dataObjects>
    <CompositeFrame id="NL:CHB:CompositeFrame:CHB" version="176">
      <ValidBetween>
        <FromDate>2026-05-05T01:32:00</FromDate>
      </ValidBetween>
      <FrameDefaults>
        <DefaultCodespaceRef ref="NL:BISON:Codespace:CHB"/>
        <DefaultDataSourceRef ref="NL:CHB:DataSource:CHB"/>
        <DefaultLocale>
          <TimeZone>Europe/Amsterdam</TimeZone>
          <DefaultLanguage>nl</DefaultLanguage>
        </DefaultLocale>
        <DefaultLocationSystem>urn:ogc:def:crs:EPSG::28992</DefaultLocationSystem>
        <DefaultSystemOfUnits>SiMetres</DefaultSystemOfUnits>
      </FrameDefaults>
      <frames>
        <ResourceFrame id="NL:CHB:ResourceFrame:

## Output

- The publisher is DOVA (<ParticipantRef>DOVA</ParticipantRef>) confirming this is the national aggregated feed

- The feed was published on May 5, 2026 (<PublicationTimestamp>2026-05-05T04:23:57.076</PublicationTimestamp>)
- The validity starts from May 5, 2026 (<FromDate>2026-05-05T01:32:00</FromDate>)

- The namespace is http://www.netex.org.uk/netex so correct
    
- The coordinate system is RD New (EPSG:28992) which seems to be  the Dutch national coordinate system, not WGS84 lat/lon. This is an important difference from Luxembourg and France where coordinates were already in standard lat/lon.
Coordinates look like <gml:pos>208889 603004</gml:pos> these are metres in the Dutch grid, not degrees



## Extracting StopPlace elements from the NeTEx file

Since the file is only 3.2 MB after decompression, it is small enough to load and parse fully into memory, no `sed` or `iterparse` needed like in France. I decompress the gzip file using `gzip.open` and parse the XML directly with `ET.parse`.

For each `StopPlace` element I extract four fields: the stop ID, the name, the transport mode, and the coordinates. The coordinates are stored as a single `<gml:pos>` string with two values separated by a space, the x and y position in the Dutch RD New coordinate system (EPSG:28992). I split these into separate `x_rd` and `y_rd` columns, which will be converted to standard WGS84 latitude and longitude in the next step.

In [18]:
netex_tag = "{http://www.netex.org.uk/netex}"
gml_tag   = "{http://www.opengis.net/gml/3.2}"

# Decompress and parse the full XML
with gzip.open(netex_path, "rb") as f:
    tree = ET.parse(f)

root = tree.getroot()

all_stopplace_rows = []
for sp in root.findall(f".//{netex_tag}StopPlace"):
    name_elem  = sp.find(f"{netex_tag}Name")
    pos_elem   = sp.find(f".//{gml_tag}pos")
    mode_elem  = sp.find(f"{netex_tag}TransportMode")

    # Coordinates are in RD New (EPSG:28992) — x y format
    x, y = None, None
    if pos_elem is not None and pos_elem.text:
        parts = pos_elem.text.strip().split()
        if len(parts) == 2:
            x, y = float(parts[0]), float(parts[1])

    all_stopplace_rows.append({
        "stopplace_id":   sp.get("id"),
        "stopplace_name": name_elem.text.strip() if name_elem is not None and name_elem.text else None,
        "transport_mode": mode_elem.text.strip() if mode_elem is not None and mode_elem.text else None,
        "x_rd":           x,
        "y_rd":           y,
    })

netex_stops_nl = pd.DataFrame(all_stopplace_rows)

print(f"Total StopPlace rows: {len(netex_stops_nl):,}")
print(f"\nMissingness:")
print(netex_stops_nl.isna().sum())
display(netex_stops_nl.head(10))

Total StopPlace rows: 25,380

Missingness:
stopplace_id        0
stopplace_name      0
transport_mode    291
x_rd                0
y_rd                0
dtype: int64


,stopplace_id,stopplace_name,transport_mode,x_rd,y_rd
0,NL:CHB:StopPlace:NL:S:lwrsgv,"Lauwersoog, Veer",ferry,208889.0,603004.0
1,NL:CHB:StopPlace:10002970,"Groningen, Protonstraat",bus,231337.0,581771.0
2,NL:CHB:StopPlace:10003290,"Groningen, Verl. Visserstraat",bus,233070.0,581932.0
3,NL:CHB:StopPlace:10006570,"Groningen, Maaslaan",bus,233702.0,580369.0
4,NL:CHB:StopPlace:10006710,"Groningen, Hippocrateslaan",bus,233167.0,579525.0
5,NL:CHB:StopPlace:10008020,"IJzendijke, Stevinstraat",bus,31522.0,372215.0
6,NL:CHB:StopPlace:10008050,"Groningen, Sontplein",bus,234808.0,581683.0
7,NL:CHB:StopPlace:10008070,"Groningen, Petrus Campersingel",bus,234574.0,581963.0
8,NL:CHB:StopPlace:10008290,"Groningen, Schuitendiep",bus,234342.0,581818.0
9,NL:CHB:StopPlace:10008570,"Groningen, Osloweg",bus,235728.0,581168.0


## Output

The extraction produced **25,380 StopPlace rows** with zero missing coordinates and zero missing IDs or names. The only missingness is **291 rows without a transport mode**, which is minor.

The StopPlace IDs follow two different patterns: some use a readable code like `NL:CHB:StopPlace:NL:S:lwrsgv` while others use a numeric ID like `NL:CHB:StopPlace:10002970`. The numeric ones look promising for matching with the GTFS `stop_id` values.

The coordinates are clearly in RD New format. Values like `208889, 603004` are metres in the Dutch national grid, not degrees. The next step is to convert these to WGS84 so they can be compared to the GTFS coordinates.

One thing to note is that **25,380 NeTEx StopPlaces vs 15,827 GTFS parent stations** is a significant difference in count unlike Luxembourg and France where the counts matched almost perfectly. This likely reflects a granularity difference where the NeTEx feed includes individual platforms and stops at a finer level than the GTFS parent station hierarchy.

## Converting RD New Coordinates to WGS84

I use the `pyproj` library to convert the coordinates. `pyproj` is a Python interface to the PROJ cartographic library, which handles coordinate transformations between any two geographic coordinate reference systems. I first create a `Transformer` object that defines the conversion from EPSG:28992 to EPSG:4326, then apply it row by row to the `x_rd` and `y_rd` columns to produce `longitude` and `latitude` values in degrees.

The `always_xy=True` parameter ensures that the output is always in the order longitude, latitude, regardless of how the coordinate system defines its axes. w

**pyproj Documentation:**
- pyproj contributors (2026). *pyproj — Cartographic transformations and geodetic computations*. https://pyproj4.github.io/pyproj/stable/

In [19]:
# Convert RD New (EPSG:28992) to WGS84 (EPSG:4326)
transformer = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)

netex_stops_nl[["longitude", "latitude"]] = netex_stops_nl.apply(
    lambda row: pd.Series(transformer.transform(row["x_rd"], row["y_rd"])),
    axis=1
)

print("Coordinate conversion done.")
print(netex_stops_nl[["latitude", "longitude"]].describe().round(4))
display(netex_stops_nl.head(10))

Coordinate conversion done.
         latitude   longitude
count  25380.0000  25380.0000
mean      52.1385      5.3954
std        0.5833      0.8343
min       50.6245      3.2143
25%       51.7859      4.7445
50%       52.0963      5.3536
75%       52.4911      6.0060
max       53.4967      7.4646


,stopplace_id,stopplace_name,transport_mode,x_rd,y_rd,longitude,latitude
0,NL:CHB:StopPlace:NL:S:lwrsgv,"Lauwersoog, Veer",ferry,208889.0,603004.0,6.197575,53.410605
1,NL:CHB:StopPlace:10002970,"Groningen, Protonstraat",bus,231337.0,581771.0,6.530016,53.217107
2,NL:CHB:StopPlace:10003290,"Groningen, Verl. Visserstraat",bus,233070.0,581932.0,6.555994,53.218304
3,NL:CHB:StopPlace:10006570,"Groningen, Maaslaan",bus,233702.0,580369.0,6.565071,53.204169
4,NL:CHB:StopPlace:10006710,"Groningen, Hippocrateslaan",bus,233167.0,579525.0,6.556861,53.196665
5,NL:CHB:StopPlace:10008020,"IJzendijke, Stevinstraat",bus,31522.0,372215.0,3.615581,51.325687
6,NL:CHB:StopPlace:10008050,"Groningen, Sontplein",bus,234808.0,581683.0,6.581947,53.215811
7,NL:CHB:StopPlace:10008070,"Groningen, Petrus Campersingel",bus,234574.0,581963.0,6.578514,53.218361
8,NL:CHB:StopPlace:10008290,"Groningen, Schuitendiep",bus,234342.0,581818.0,6.575005,53.217093
9,NL:CHB:StopPlace:10008570,"Groningen, Osloweg",bus,235728.0,581168.0,6.595587,53.211046


## Output

The coordinate conversion worked perfectly. All 25,380 StopPlaces now have valid WGS84 coordinates. The latitude values range from **50.62 to 53.50** and longitude from **3.21 to 7.46**, which is exactly the correct geographic range for the Netherlands. The mean coordinates (52.14°N, 5.40°E) place the centre of the feed right in the middle of the country, as expected.


## Extract lines

In [20]:
# Check what tags exist in the Netex file (full scan, not a partial sample)
tag_counts = {}
with gzip.open(netex_path, "rb") as f:
    for event, elem in ET.iterparse(f, events=("start",)):
        tag = elem.tag.split("}")[-1]
        tag_counts[tag] = tag_counts.get(tag, 0) + 1
        elem.clear()

print(f"Total tags scanned: {sum(tag_counts.values()):,}\n")

for key in ["StopPlace", "Quay", "Line", "Route", "ServiceJourney", "DayType", "DayTypeAssignment", "UicOperatingPeriod", "OperatingPeriod"]:
    print(f"{key}: {tag_counts.get(key, 0):,}")

print("\nTop 20 tags overall:")
for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1])[:20]:
    print(f"{count:>8} {tag}")

Total tags scanned: 1,191,031

StopPlace: 25,380
Quay: 49,442
Line: 0
Route: 0
ServiceJourney: 0
DayType: 0
DayTypeAssignment: 0
UicOperatingPeriod: 0
OperatingPeriod: 0

Top 20 tags overall:
   75973 AccessibilityAssessment
   75973 MobilityImpairedAccess
   75525 Centroid
   75525 Location
   75525 pos
   74822 privateCodes
   74822 PrivateCode
   49442 Quay
   49442 QuayType
   49166 limitations
   49166 AccessibilityLimitation
   49166 WheelchairAccess
   49166 StepFreeAccess
   49166 LiftFreeAccess
   49166 RampFreeAccess
   49166 LevelAccessIntoVehicle
   49166 TactileGuidanceAvailable
   26680 Name
   26213 TransportMode
   25380 StopPlace


## Comment

The tag check reveals that the DOVA EPIP file contains **only stop-related elements** like `StopPlace`, `Quay`, `Name`, `TransportMode`, `Centroid` and accessibility fields. There are no `Line`, `Route`, `DayType`, `UicOperatingPeriod` or `ServiceJourney` tags present, confirming that this file is purely a stop registry and cannot be used for line or calendar comparison.

<div style="background-color: #f8d7da; border-left: 5px solid #dc3545; padding: 16px 20px; border-radius: 4px; margin: 10px 0;">
<h2 style="margin-top: 0;"> Important Finding: Structure of the Netherlands NeTEx Data</h2>

The Netherlands NeTEx data available on NDOV Loket is structured very differently from Luxembourg and France. Rather than a single national NeTEx file covering stops, lines, and calendar data, the Netherlands splits its NeTEx data across two separate layers:

<br><br>

<b>1. DOVA EPIP — National Stop Registry</b><br>
The <code>epiap</code> folder contains a single national NeTEx file (<code>NeTEx_DOVA_epiap</code>) published by DOVA (Dienst OV-aanbesteding), the national public transport data authority. This file covers all Dutch operators but contains <b>only stop and station data</b>, so no lines, no timetables, no calendar information.

<br><br>

<b>2. Per-operator timetable files</b><br>
Individual operators publish their own NeTEx timetable files in separate folders (arr, cxx, gvb, htm, qbuzz, ret, etc.). However, <b>NS (Nederlandse Spoorwegen)</b>, the national rail operator, does not publish NeTEx timetable data on NDOV Loket. Instead, NS distributes its timetable in <b>IFF format</b> which is an older Dutch-specific proprietary format developed in the 1990s that is not an international standard.

<br><br>

This fragmentation means that a full GTFS vs NeTEx comparison at the line and calendar level, as performed for other countries, is <b>not possible with freely available data</b> for the Netherlands. The comparison for the Netherlands will therefore be limited to the <b>station level only</b>, using the DOVA EPIP NeTEx file for the NeTEx side and the OVapi GTFS feed for the GTFS side.

<br><br>

This structural difference is itself a relevant finding, as it highlights the uneven implementation of NeTEx across European countries.
</div>

# Part 3: station-level comparison between GTFS and NeTEx

## Checking for ID-based Matching Potential

Before deciding on a matching strategy, I first check whether the NeTEx `stopplace_id` values contain a numeric part that could be matched directly to GTFS `stop_id` values. I extract the numeric part at the end of each NeTEx ID using a regular expression.

In [21]:
# Check if NeTEx numeric IDs overlap with GTFS stop_ids
netex_stops_nl["extracted_id"] = (
    netex_stops_nl["stopplace_id"]
    .str.extract(r":(\d+)$")
)

print("NeTEx IDs with extractable numeric part:", netex_stops_nl["extracted_id"].notna().sum())
print("\nSample:")
print(netex_stops_nl[["stopplace_id", "extracted_id"]].head(10).to_string(index=False))

NeTEx IDs with extractable numeric part: 25371

Sample:
                stopplace_id extracted_id
NL:CHB:StopPlace:NL:S:lwrsgv          NaN
   NL:CHB:StopPlace:10002970     10002970
   NL:CHB:StopPlace:10003290     10003290
   NL:CHB:StopPlace:10006570     10006570
   NL:CHB:StopPlace:10006710     10006710
   NL:CHB:StopPlace:10008020     10008020
   NL:CHB:StopPlace:10008050     10008050
   NL:CHB:StopPlace:10008070     10008070
   NL:CHB:StopPlace:10008290     10008290
   NL:CHB:StopPlace:10008570     10008570


## Comment

**25,371 out of 25,380** NeTEx StopPlaces have an extractable numeric ID. 

Only 9 have a non-numeric format (like `NL:CHB:StopPlace:NL:S:lwrsgv`). 

The extracted IDs look like `10002970`, `10003290` etc.

## Checking ID Overlap Between GTFS and NeTEx

I apply the same numeric extraction to the GTFS `stop_id` column and then check how many IDs appear in both feeds. If there is a strong overlap, I can use ID-based matching. If not, I stick to coordinate-based matching.

In [22]:
# Extract numeric part from GTFS stop_id as well
gtfs_station_stops["extracted_id"] = (
    gtfs_station_stops["stop_id"]
    .str.extract(r":(\d+)$")
)

print("GTFS IDs with extractable numeric part:", gtfs_station_stops["extracted_id"].notna().sum())
print("\nSample:")
print(gtfs_station_stops[["stop_id", "extracted_id"]].head(10).to_string(index=False))

# Check overlap
gtfs_ids          = set(gtfs_station_stops["extracted_id"].dropna().astype(str))
netex_numeric_ids = set(netex_stops_nl["extracted_id"].dropna().astype(str))

print(f"\nGTFS numeric IDs:                    {len(gtfs_ids):,}")
print(f"NeTEx numeric IDs:                   {len(netex_numeric_ids):,}")
print(f"IDs present in both:                 {len(gtfs_ids & netex_numeric_ids):,}")
print(f"Only in GTFS:                        {len(gtfs_ids - netex_numeric_ids):,}")
print(f"Only in NeTEx:                       {len(netex_numeric_ids - gtfs_ids):,}")

GTFS IDs with extractable numeric part: 15827

Sample:
        stop_id extracted_id
stoparea:109705       109705
stoparea:109707       109707
stoparea:109708       109708
stoparea:109709       109709
stoparea:109711       109711
stoparea:109712       109712
stoparea:109714       109714
stoparea:109715       109715
stoparea:111940       111940
stoparea:116635       116635

GTFS numeric IDs:                    15,827
NeTEx numeric IDs:                   25,371
IDs present in both:                 0
Only in GTFS:                        15,827
Only in NeTEx:                       25,371


## Observation

The result is clear: **0 IDs match** between the two feeds. All 15,827 GTFS IDs are only in GTFS, and all 25,371 NeTEx IDs are only in NeTEx.

Looking at the ID formats, the reason is obvious because GTFS uses 6-digit IDs like `109705` while NeTEx uses 8-digit IDs like `10002970`. They are completely different numbering systems with no overlap.

This means **ID-based matching is not possible** for the Netherlands. I will stick to **coordinate-based matching**. 

## Station Matching: KDTree Approach

Instead of the haversine-based loop used for Luxembourg and France, here I use a **KDTree** (K-Dimensional Tree) from the `scipy` library. The reason is that with 15,827 GTFS stations and 25,380 NeTEx StopPlaces, the haversine loop would need to compute roughly 400 million individual distance calculations, which takes too long in this environment.

A KDTree is a data structure that organises points in space so that nearest neighbour searches can be done in a fraction of the time instead of checking every possible pair, it intelligently narrows down the candidates. In practice, the query runs in seconds rather than minutes.

The trade-off is that KDTree works with **degrees** (the raw coordinate values) rather than metres, so the distances it returns are in degrees rather than real-world distance. To convert them to approximate metres, I multiply by 111,000 (the number of metres in one degree of latitude at the equator). This approximation is accurate enough for the Netherlands, which is at a mid latitude and covers a relatively small geographic area.

In [23]:
# Prepare coordinates 

# Ensure coordinates are numeric on both sides
gtfs_station_stops["stop_lat"] = pd.to_numeric(gtfs_station_stops["stop_lat"], errors="coerce")
gtfs_station_stops["stop_lon"] = pd.to_numeric(gtfs_station_stops["stop_lon"], errors="coerce")
netex_stops_nl["latitude"]     = pd.to_numeric(netex_stops_nl["latitude"],  errors="coerce")
netex_stops_nl["longitude"]    = pd.to_numeric(netex_stops_nl["longitude"], errors="coerce")

# Drop rows with invalid coordinates on either side
gtfs_valid  = gtfs_station_stops[
    gtfs_station_stops["roughly_in_netherlands"] == True
].dropna(subset=["stop_lat", "stop_lon"]).copy()

netex_valid = netex_stops_nl.dropna(subset=["latitude", "longitude"]).copy()

print(f"GTFS stations for matching:    {len(gtfs_valid):,}")
print(f"NeTEx StopPlaces for matching: {len(netex_valid):,}")

# Build NeTEx coordinate list for fast lookup
netex_coords = netex_valid[["latitude", "longitude"]].values.tolist()
netex_ids    = netex_valid["stopplace_id"].tolist()
netex_names  = netex_valid["stopplace_name"].tolist()

print("Coordinate lists ready.")

GTFS stations for matching:    15,658
NeTEx StopPlaces for matching: 25,380
Coordinate lists ready.


In [24]:
# Build a KDTree from NeTEx coordinates
netex_array = np.array(netex_coords)
gtfs_array  = np.array([(row.stop_lat, row.stop_lon) for _, row in gtfs_valid.iterrows()])

netex_kdtree = cKDTree(netex_array)

# Find nearest NeTEx stop for each GTFS station
distances, indices = netex_kdtree.query(gtfs_array, k=1)

# Build results
results = []
for i, (_, gtfs_row) in enumerate(gtfs_valid.iterrows()):
    best_idx = indices[i]
    results.append({
        "stop_id":        gtfs_row["stop_id"],
        "stop_name":      gtfs_row["stop_name"],
        "stopplace_id":   netex_ids[best_idx],
        "stopplace_name": netex_names[best_idx],
        "distance_deg":   distances[i],
    })

print("Matching done!")

Matching done!


In [25]:
station_match_nl = pd.DataFrame(results)

# Convert degrees to approximate metres (1 degree ≈ 111,000 metres)
station_match_nl["distance_m"] = station_match_nl["distance_deg"] * 111000

print(f"Total GTFS stations:       {len(station_match_nl):,}")
print(f"Matched within 50m:        {(station_match_nl['distance_m'] <= 50).sum():,}")
print(f"Matched within 100m:       {(station_match_nl['distance_m'] <= 100).sum():,}")
print(f"Unmatched (beyond 100m):   {(station_match_nl['distance_m'] > 100).sum():,}")

Total GTFS stations:       15,658
Matched within 50m:        14,536
Matched within 100m:       15,235
Unmatched (beyond 100m):   423


## Result

The station-level comparison shows strong practical overlap between the Netherlands GTFS and NeTEx stop data. Using a strict 50 m threshold, **14,536 of 15,658 GTFS stations match a NeTEx StopPlace, corresponding to 92.8%**. With a wider 100 m threshold, 15,235 stations match, leaving 423 stations beyond 100 m.

The unmatched examples are mostly cross-border or international stops from Belgium and Germany that are present in the Dutch GTFS feed but not covered in the Dutch DOVA EPIP NeTEx stop registry. For domestic Dutch stops, the overlap appears very strong.

In [26]:
matched   = station_match_nl[station_match_nl["distance_m"] <= 50]
unmatched = station_match_nl[station_match_nl["distance_m"] > 100]

print(f"Matched within 50m: {len(matched):,}")
print(f"Unmatched beyond 100m: {len(unmatched):,}")

print("\n--- Sample matched ---")
display(matched.head(10))

print("\n--- Sample unmatched ---")
display(unmatched.head(10))

Matched within 50m: 14,536
Unmatched beyond 100m: 423

--- Sample matched ---


,stop_id,stop_name,stopplace_id,stopplace_name,distance_deg,distance_m
1,stoparea:109708,Nijmegen Goffert,NL:CHB:StopPlace:8400477,Nijmegen Goffert,0.000012,1.277986
6,stoparea:111940,Eindhoven Stadion,NL:CHB:StopPlace:8499998,Eindhoven Stadion,0.000004,0.410689
21,stoparea:153154,De Westereen,NL:CHB:StopPlace:21380800,"De Westereen, Station De Westereen",0.000283,31.401433
27,stoparea:165459,Dronryp,NL:CHB:StopPlace:8400192,Dronryp,0.000126,13.955840
29,stoparea:17775,Holten,NL:CHB:StopPlace:8400328,Holten,0.000328,36.428465
30,stoparea:17776,Almere Poort,NL:CHB:StopPlace:8400450,Almere Poort,0.000048,5.313429
31,stoparea:17777,Hollandsche Rading,NL:CHB:StopPlace:8400327,Hollandsche Rading,0.000354,39.248551
35,stoparea:17781,Duivendrecht,NL:CHB:StopPlace:8400194,Duivendrecht,0.000045,4.966674
39,stoparea:17784,Duiven,NL:CHB:StopPlace:41251110,"Duiven, Station Duiven",0.000178,19.727127
40,stoparea:17785,Nieuw Vennep,NL:CHB:StopPlace:8400460,Nieuw Vennep,0.000032,3.497028



--- Sample unmatched ---


,stop_id,stop_name,stopplace_id,stopplace_name,distance_deg,distance_m
0,stoparea:109705,Tongeren,NL:CHB:StopPlace:66580350,"Maastricht, Capitoolhof",0.182265,20231.400805
2,stoparea:109709,Glons,BE:CHB:StopPlace:8841004,Liège-Guilemins,0.129956,14425.144639
3,stoparea:109711,Bilzen,NL:CHB:StopPlace:66581740,"Maastricht, Pierebolruwe",0.138566,15380.831384
4,stoparea:109714,Diepenbeek,NL:CHB:StopPlace:66581740,"Maastricht, Pierebolruwe",0.233213,25886.687753
5,stoparea:109715,Hasselt,BE:CHB:StopPlace:64621280,"Lommel, Station",0.281444,31240.237136
8,stoparea:124586,Eilendorf,DE:CHB:StopPlace:66467400,"Aachen, Kaiserplatz",0.060377,6701.840134
9,stoparea:124588,Aachen Rothe Erde,DE:CHB:StopPlace:66467410,"Aachen, Kaiserplatz",0.021784,2417.975006
10,stoparea:124590,Stolberg Rathaus,DE:CHB:StopPlace:66467410,"Aachen, Kaiserplatz",0.134939,14978.225368
11,stoparea:124591,Eschweiler-Nothberg,DE:CHB:StopPlace:66467400,"Aachen, Kaiserplatz",0.202130,22436.476197
12,stoparea:124592,Eschweiler-West,DE:CHB:StopPlace:66467400,"Aachen, Kaiserplatz",0.162306,18015.913868


## Conclusion

The station-level comparison shows strong practical overlap between the Netherlands GTFS and NeTEx stop data. Using a strict 50 m threshold, 14,536 of 15,658 GTFS stations match a NeTEx StopPlace (92.8%); with a wider 100 m threshold, 15,235 match, leaving 423 stations beyond 100 m. These two thresholds are reported separately since they answer slightly different questions -- 92.8% is not "100% minus the 423 unmatched," those 423 are the beyond-100m count specifically.

The unmatched stations are not a data quality issue -- they are mostly cross-border or international stops from Belgium and Germany that are present in the Dutch GTFS feed but not covered by the Dutch DOVA EPIP NeTEx stop registry. For domestic Dutch stations, both feeds are fully consistent.

Unlike Luxembourg and France, the comparison had to rely on **coordinate-based matching**, since GTFS and NeTEx use completely different ID systems here. The Netherlands NeTEx file is also limited to stop-place information -- a full scan confirms zero `Line`, `Route`, `ServiceJourney`, `DayType`, `DayTypeAssignment`, or `OperatingPeriod` tags anywhere in the file -- so line-level and calendar-level comparisons could not be performed from this file.